In [1]:
import csv
import pandas as pd

In [2]:
# Dataframes
base_path ="data/company_office/"
df_companies = pd.read_csv(base_path + 'companies.csv')
df_office_locations = pd.read_csv(base_path + 'office_locations.csv')
df_buildings = pd.read_csv(base_path + 'buildings.csv')


In [3]:
import duckdb

In [4]:
base_path ="data/company_office/"
duckdb.read_csv(base_path + 'companies.csv')
duckdb.read_csv(base_path + 'office_locations.csv')
duckdb.read_csv(base_path + 'buildings.csv')



┌───────┬─────────────────────────────────────┬─────────────┬────────┬─────────┬────────────────────┐
│  id   │                name                 │    City     │ Height │ Stories │       Status       │
│ int64 │               varchar               │   varchar   │ int64  │  int64  │      varchar       │
├───────┼─────────────────────────────────────┼─────────────┼────────┼─────────┼────────────────────┤
│     1 │ Torre KOI                           │ Monterrey   │    220 │      67 │ under construction │
│     2 │ Torre Mitikah                       │ Mexico City │    210 │      60 │ under construction │
│     3 │ Punto Chapultepec                   │ Mexico City │    210 │      59 │ proposed           │
│     4 │ Torre Reforma                       │ Mexico City │    330 │      57 │ under construction │
│     5 │ Corporativo BBVA Bancomer           │ Mexico City │    220 │      50 │ under construction │
│     6 │ Reforma 432                         │ Mexico City │    300 │     100 │ u

In [5]:
try:
    duckdb.sql("CREATE TABLE companies AS SELECT * FROM 'data/company_office/companies.csv';")
except:
    print("Table 'companies' already exists. Skipping creation.")
    pass

try:
    duckdb.sql("CREATE TABLE office_locations AS SELECT * FROM 'data/company_office/office_locations.csv';")
except:
    print("Table 'office_locations' already exists. Skipping creation.")
    pass

try:
    duckdb.sql("CREATE TABLE buildings AS SELECT * FROM 'data/company_office/buildings.csv';")
except:
    print("Table 'buildings' already exists. Skipping creation.")
    pass

In [23]:
schema_info = duckdb.sql("PRAGMA table_info('companies');").df().to_string(index=False)


In [18]:
import ollama

# 1. FUNÇÃO PARA EXTRAIR O SCHEMA AUTOMATICAMENTE
def obter_schema_completo():
    # Descobre todas as tabelas guardadas no teu DuckDB
    tabelas = duckdb.sql("SHOW TABLES;").df()['name'].tolist()
    
    schema_texto = "Tens acesso às seguintes tabelas e colunas no DuckDB:\n"
    
    # Para cada tabela, extrai as colunas e os tipos de dados
    for tabela in tabelas:
        schema_texto += f"\n- Tabela: '{tabela}'\n  Colunas:\n"
        # Usamos o PRAGMA que estavas a tentar usar, mas aplicável a todas as tabelas
        info_colunas = duckdb.sql(f"PRAGMA table_info('{tabela}');").df()
        
        for _, row in info_colunas.iterrows():
            schema_texto += f"    * {row['name']} ({row['type']})\n"
            
    return schema_texto

# Gerar o contexto atualizado com 'companies', 'office_locations' e 'buildings'
contexto_tabelas = obter_schema_completo()

# 2. SEPARAÇÃO DAS REGRAS DO AGENTE (SYSTEM PROMPT)
system_prompt = """
Tu és um agente automatizado exclusivo para DuckDB. A tua única função é gerar código SQL.
REGRAS ESTRITAS:
1. Responde APENAS com o bloco de código SQL em formato Markdown (```sql ... ```).
2. Não incluas NENHUM texto explicativo, saudação ou notas de rodapé.
3. Usa a sintaxe nativa do DuckDB sempre que apropriado.
"""

# 3. PEDIDO DO UTILIZADOR (Podes mudar esta frase para o que quiseres)
pergunta_utilizador = "Quero ver o nome dos edifícios e o status que estão na cidade de 'Mexico City'."

# Juntamos o contexto gerado automaticamente com o pedido
prompt_final = f"{contexto_tabelas}\nPedido do Utilizador: {pergunta_utilizador}"

# 4. CHAMADA AO OLLAMA (Usando chat para separar System de User)
result = ollama.chat(
    model='llama3', # Recomendo vivamente atualizar o 'llama2' para o 'llama3' ou 'codellama'
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_final}
    ]
)

print(result['message']['content'])

```sql
SELECT name, Status 
FROM buildings 
WHERE City = 'Mexico City'
```


In [21]:

import re

result_string = result['message']['content']
match = re.search(r"```sql\s+(.*?)\s+```", result_string, re.DOTALL)
if match:
    clean_query = match.group(1)
else:
    # Caso o modelo se tenha esquecido das tags por algum motivo,
    # removemos apenas possíveis espaços ou quebras de linha extra
    clean_query = result_string.strip()

print("Query Limpa Pronta a Usar:")
print(clean_query)
print("-" * 40)

Query Limpa Pronta a Usar:
SELECT name, Status 
FROM buildings 
WHERE City = 'Mexico City'
----------------------------------------


In [25]:
query = duckdb.sql(clean_query).df().to_string(index=False)
print("Resultado da Query:")
print(query)

Resultado da Query:
                               name             Status
                      Torre Mitikah under construction
                  Punto Chapultepec           proposed
                      Torre Reforma under construction
          Corporativo BBVA Bancomer under construction
                        Reforma 432 under construction
                Torre New York Life under construction
Residencial Vidalta Torre Altaire 2            on-hold
Residencial Vidalta Torre Altaire 3            on-hold
                         Reforma 90            on-hold
           Ritz-Carlton Mexico City            on-hold


primeiro vou ligar o modelo depois vou fazer a query e depois vou fazer o merge dos dataframes